# Práctica guiada — De formulario a experiencia de inferencia

En S5 ya construisteis un formulario Streamlit que recoge los 11 atributos y llama a un gateway basado en el contrato de S4. En S6 **no se rediseña el contrato ni se vuelve a implementar el formulario**. El objetivo es convertir esa pantalla funcional en una interfaz operable: mantiene el resultado al haber un rerun, hace visible lo que está ocurriendo y enseña la información justa para interpretar e investigar una predicción.

Trabajad en parejas. Completad los `TODO`, ejecutad las comprobaciones y conservad el notebook como evidencia para el taller. La implementación queda para la Clase 2.

## Entrega de la práctica

Al terminar debéis poder enseñar cuatro decisiones concretas:

1. qué permanece en la sesión, qué se cachea y qué no se retiene;
2. la máquina `idle → loading → success/error`;
3. un plano de pantalla con componentes Streamlit concretos;
4. mensajes seguros para confianza baja, lentitud y error.

No se entrega código de inferencia nuevo. S4 sigue siendo la única frontera de validación y predicción.

## 1. Comprobar el punto de partida de S5

Ejecutad la celda. Debe demostrar que partís del formulario y gateway de S5, no de una segunda implementación. Después anotad una limitación que observaríais al cambiar un widget o al enviar una segunda predicción.

In [1]:
from pathlib import Path

repo_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / 'semana5/modules/05-streamlit-basic-model-ui').is_dir()
)
s5_app = repo_root / 'semana5/modules/05-streamlit-basic-model-ui/solutions/02-first-streamlit/app.py'
source = s5_app.read_text(encoding='utf-8')

checks = {
    'reutiliza st.form': 'st.form(' in source,
    'recoge valores antes de inferir': 'collect_values' in source,
    'aún no usa session_state': 'st.session_state' not in source,
}
print('app de S5:', s5_app)
for label, passed in checks.items():
    print(f"{'✓' if passed else '✗'} {label}")
assert all(checks.values())

# TODO: escribid aquí una limitación observable de S5.
limitacion_s5 = (
    'El resultado vive en una variable local: al tocar cualquier widget, '
    'Streamlit re-ejecuta el script y la predicción desaparece. Además, el '
    'bundle se vuelve a cargar del disco en cada rerun y no hay loading, '
    'reintento ni contadores.'
)
print('Limitación observada:', limitacion_s5)

app de S5: semana5/modules/05-streamlit-basic-model-ui/solutions/02-first-streamlit/app.py
✓ reutiliza st.form
✓ recoge valores antes de inferir
✓ aún no usa session_state
Limitación observada: El resultado vive en una variable local: al tocar cualquier widget, Streamlit re-ejecuta el script y la predicción desaparece. Además, el bundle se vuelve a cargar del disco en cada rerun y no hay loading, reintento ni contadores.


## 2. Decidir dónde vive cada cosa

Un rerun vuelve a ejecutar el script. Una variable local desaparece; `st.session_state` conserva información de **esa sesión**; `st.cache_resource` reutiliza recursos como un gateway o modelo. Un recurso cacheado global puede compartirse entre usuarios, así que debe ser seguro para concurrencia.

Completad la tabla y justificad cada decisión. No guardéis secretos ni las 11 features en la telemetría. Los valores del último envío solo se conservan para permitir un reintento y no se muestran ni se registran.

| Clave o recurso | Dónde vive | ¿Sobrevive a un rerun? | ¿Se limpia al pulsar «Limpiar»? | Motivo |
| --- | --- | --- | --- | --- |
| `last_state` (`UiState`) | `st.session_state` | Sí | **Sí**, vuelve a `idle` | Es lo que se vuelve a pintar tras cada rerun. Pertenece a la sesión de esta persona. |
| `last_values` (`dict`) | `st.session_state` | Sí | **Sí**, se vacía | Solo sirve para «Reintentar» el último envío. No se muestra ni se registra. |
| `telemetry` (`Telemetry`) | `st.session_state` | Sí | **No** | Son contadores de toda la sesión (peticiones, errores, latencias, versiones). Limpiar un resultado no borra la historia. |
| gateway / bundle | `st.cache_resource` | Sí, y se comparte entre sesiones | No | Cargar el modelo es caro: se hace una vez y se reutiliza. Debe ser de solo lectura. |
| respuesta de una petición | **No** se cachea; solo se guarda la vista en `last_state` | Sí, dentro de `last_state` | Sí | Es un dato de una persona. Si estuviera en `cache_resource`, otra sesión podría ver el resultado ajeno. |

**Decisión de caché:** se envuelve con `st.cache_resource` la función que construye el gateway (`build_gateway(MODEL_UI_BUNDLE)`). El argumento de la caché es la **ruta del bundle**: si cambia la ruta, se carga otro gateway. **No** se cachea ni el payload de entrada ni la predicción, porque son datos de una petición concreta.

**Telemetría:** solo agregados (número de peticiones, éxitos, errores por código, latencia media y p95, versiones). **Nunca** los valores de las 11 features.


## 3. Diseñar la pantalla, no solo el estado

La interfaz debe tener cuatro zonas, en este orden:

1. **acción:** el formulario heredado de S5;
2. **operación:** qué está ocurriendo mientras se ejecuta;
3. **resultado:** categoría, confianza y latencia;
4. **trazabilidad:** versiones y `request_id`, visibles solo si se necesitan.

Elegid componentes que tengan una función. `st.columns` organiza, `st.status` comunica una operación, `st.metric` destaca un dato y `st.expander` aplaza el detalle técnico. No uséis una barra de progreso para la confianza: representa avance de una tarea, no certeza del modelo.

In [2]:
# Sustituid cada TODO por el componente y la justificación que usaríais.
# Las claves describen zonas de la interfaz; no son nuevos contratos de ML.

ui_plan = {
    'formulario': {
        'component': 'st.form',
        'reason': 'Se hereda de S5: editar no ejecuta el modelo, solo el botón.',
    },
    'estado_operacion': {
        'component': 'st.status',
        'reason': 'Muestra que la petición está en curso sin ocultar el formulario.',
    },
    'resumen_resultado': {
        'component': 'st.metric',
        'reason': 'Categoría, confianza y latencia se leen de un vistazo, en 3 columnas.',
    },
    'mensaje_interpretacion': {
        'component': 'st.warning',
        'reason': 'Avisa de confianza baja o lentitud; con confianza normal, st.info.',
    },
    'trazabilidad': {
        'component': 'st.expander',
        'reason': 'Versiones y request_id quedan disponibles sin invadir el resultado.',
    },
    'separacion_telemetria': {
        'component': 'st.divider',
        'reason': 'Separa los agregados de la sesión del resultado de una petición.',
    },
}

allowed_components = {
    'st.form', 'st.status', 'st.empty', 'st.columns', 'st.container',
    'st.metric', 'st.success', 'st.info', 'st.warning', 'st.error',
    'st.expander', 'st.divider', 'st.caption',
}
required_zones = {
    'formulario', 'estado_operacion', 'resumen_resultado',
    'mensaje_interpretacion', 'trazabilidad', 'separacion_telemetria',
}

assert set(ui_plan) == required_zones
for zone, decision in ui_plan.items():
    assert set(decision) == {'component', 'reason'}, zone

pending = [zone for zone, decision in ui_plan.items() if 'TODO' in decision.values()]
invalid = [
    f"{zone}: {decision['component']}"
    for zone, decision in ui_plan.items()
    if 'TODO' not in decision['component'] and decision['component'] not in allowed_components
]
print('Zonas pendientes:', pending or 'ninguna')
print('Componentes no permitidos:', invalid or 'ninguno')
print('Comprobación final: pending == [] and invalid == []')

Zonas pendientes: ninguna
Componentes no permitidos: ninguno
Comprobación final: pending == [] and invalid == []


### Plano mínimo esperado

```text
Título y contexto breve

[ formulario de S5 en dos columnas ]

[ estado de la operación ]

[ Categoría ]  [ Confianza ]  [ Latencia ]
mensaje interpretativo de confianza y latencia
▶ Trazabilidad: versiones y request_id

────────────────────────────────────
Telemetría agregada de la sesión
```

Podéis entregar este plano en el notebook o redibujarlo en draw.io. Justificad cualquier cambio respecto a esta estructura.

## 4. Máquina de estados y recuperación

Completad la tabla. Una respuesta válida con latencia superior a 300 ms sigue en `success`: añade un aviso técnico, no se convierte en error. La confianza es una señal para revisar, no un diagnóstico ni una garantía.

| Estado | Evento de entrada | Zona de pantalla que cambia | Qué se muestra | Acción permitida | Evidencia |
| --- | --- | --- | --- | --- | --- |
| `idle` | Inicio / «Limpiar» | Resultado | `st.info`: «Completa el formulario y pulsa «Ejecutar inferencia».» | Editar y enviar | Captura de inicio; `last_state.phase == "idle"` |
| `loading` | Pulsar «Ejecutar inferencia» | Operación | `st.status` «Ejecutando inferencia…» | Esperar (el formulario sigue visible) | El controlador emite `loading` antes de llamar al gateway (test del controlador) |
| `success` | Payload válido | Resultado + trazabilidad | 3 métricas (categoría, confianza, latencia), mensaje de confianza y, si tarda más de 300 ms, aviso de lentitud. Versiones y `request_id` en el expander | Editar y enviar otra vez, «Limpiar» | Captura de éxito; la telemetría suma 1 éxito y la versión |
| `error` | Excepción controlada | Resultado | `st.error` con título y mensaje, `st.info` con la recuperación y `request_id` | «Reintentar» (reutiliza `last_values`), «Limpiar», corregir y enviar | Captura de error; la telemetría suma 1 error por código |

**Recuperación por tipo de error:**

| Error | Código | Qué ve la persona | Recuperación |
| --- | --- | --- | --- |
| Entrada inválida | `invalid_input` | «Datos no válidos» | Revisar el formulario y volver a enviar |
| Bundle no disponible | `artifact_unavailable` | «Modelo no disponible» | Revisar `MODEL_UI_BUNDLE` y volver a abrir la app |
| Timeout | `timeout` | «La inferencia tardó demasiado» | Esperar y pulsar «Reintentar» |
| Otro fallo del modelo | `prediction_error` | «No se pudo obtener una predicción» | «Reintentar»; si persiste, avisar con el `request_id` |

**Lo que la persona nunca debe ver:** traceback, rutas internas (`C:\...\model.joblib`), secretos ni el payload completo.

**Respuesta lenta:** una salida válida con más de 300 ms sigue en `success`. Solo se añade un aviso técnico de lentitud; no es un error del modelo.


## 5. Escribir el copy de una interfaz responsable

Cada mensaje debe decir qué ha ocurrido, qué significa y qué puede hacer la persona. No debéis prometer que una predicción es correcta. Completad textos de una frase, sin terminología interna.

In [3]:
ux_copy = {
    'low_confidence': (
        'Confianza baja: el resultado es orientativo y requiere revisión; '
        'la confianza no es una garantía.'
    ),
    'slow_success': (
        'La respuesta llegó, pero la latencia supera el objetivo técnico de la demo.'
    ),
    'invalid_input': (
        'Algún valor no cumple el contrato de entrada: revisa el formulario '
        'y vuelve a enviarlo.'
    ),
    'artifact_unavailable': (
        'No se pudo cargar el modelo configurado: comprueba MODEL_UI_BUNDLE '
        'y vuelve a abrir la aplicación.'
    ),
}

assert set(ux_copy) == {
    'low_confidence', 'slow_success', 'invalid_input', 'artifact_unavailable'
}
for scenario, message in ux_copy.items():
    assert isinstance(message, str) and message.strip(), scenario

pending_copy = [scenario for scenario, message in ux_copy.items() if 'TODO' in message]
print('Mensajes pendientes:', pending_copy or 'ninguno')
print('Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?')

Mensajes pendientes: ninguno
Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?


## Decisión visual en el laboratorio (`ui_playground.py`)

Hemos revisado los cinco escenarios del laboratorio (`Sin petición`, `Ejecutando`, `Éxito — confianza intermedia`, `Éxito — confianza baja y lento` y `Error recuperable`).

**Cambio:** en el escenario de éxito hemos quitado el `st.success(quality_label)` que aparecía encima de las métricas.

**Problema que resuelve:** ese recuadro repetía la categoría que ya muestra la métrica «Resultado» y siempre la pintaba en **verde**, incluso con «Requiere revisión» y confianza baja. El verde transmite «todo correcto, aprobado», que es justo lo que no debe comunicar un resultado orientativo. Ahora la categoría aparece una sola vez (en la métrica) y el color lo pone el mensaje de interpretación: amarillo (`st.warning`) si la confianza es baja o la respuesta es lenta, y azul (`st.info`) si no.

No se ha tocado ninguna constante de ejemplo ni el contrato de S4.

## Plan de implementación para la práctica 6.2

| Requisito | Función | Test que lo comprueba |
| --- | --- | --- |
| Crear `last_state`, `telemetry` y `last_values` solo si no existen | `session.initialize_session_state` | `test_session_state_initialization_is_idempotent` |
| «Limpiar» vuelve a `idle` sin borrar la telemetría | `session.clear_last_state` | `test_clear_last_state_keeps_telemetry` |
| Confianza: < 0,60 baja, < 0,85 media, resto alta; fuera de [0, 1] es error | `policies.classify_confidence` | `test_confidence_policy_has_explicit_boundary_values` |
| Latencia: ≤ 300 ms dentro del objetivo; > 300 ms por encima | `policies.classify_latency` | `test_latency_and_confidence_are_explained_to_the_user` |
| Convertir la predicción en textos para la pantalla | `presentation.build_prediction_view` | `test_latency_and_confidence_are_explained_to_the_user` |
| Traducir errores a código, mensaje y recuperación, sin detalles internos | `presentation.to_user_facing_error` | `test_error_translation_does_not_expose_technical_detail` |
| `loading` → medir → validar → `success`/`error` y registrar agregados | `controller.PredictionController.submit` | `test_controller_emits_loading_then_success_and_records_versions` y `..._error_without_payload_telemetry` |
| Cargar el gateway una sola vez | `app.get_cached_gateway` (`st.cache_resource`) | Revisión visual en la app |
| Pintar `idle`/`loading`/`success`/`error` | `app.render_state` | Revisión visual en la app (capturas) |

## 6. Preparar el taller

Entregad el notebook con los TODO completados. Antes de cerrar, otra pareja debe poder responder estas preguntas leyendo vuestro trabajo:

- ¿Qué se mantiene al pulsar otro widget y qué se elimina con «Limpiar último resultado»?
- ¿Qué función cargará el gateway una sola vez y cuál será su límite?
- ¿Qué componentes dibujará `render_state()` para `loading`, `success` y `error`?
- ¿Qué ve una persona con confianza baja o una respuesta lenta?
- ¿Dónde aparecen el `request_id` y las versiones sin invadir el resultado?

En el taller se implementarán estas decisiones en `session.py`, `policies.py`, `presentation.py`, `controller.py` y `app.py`.

## Debrief

- ¿Qué limita a S5 aunque el formulario ya funcione?
- ¿Por qué el gateway es un recurso cacheable y el resultado de una petición pertenece a una sesión?
- ¿Qué componente hace visible cada transición sin añadir una segunda lógica de inferencia?
- ¿Por qué categoría, confianza y latencia no tienen el mismo significado?
- ¿Qué parte de este diseño se conserva al sustituir el gateway local por HTTP en S7?